<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Machine Learning Fundamentals** · Week 2 · Trees, Ensembles and More · Lesson 11 of 24

# Naive Bayes: Fast Probability Classifiers

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals).

**Learning objective:** Explain how Naive Bayes uses Bayes' theorem with a "naive" independence assumption, and build a small SMS spam filter with scikit-learn.

## Bayes' theorem again

In Math & Python for AI you used Bayes' theorem to update a belief with evidence. **Naive Bayes** turns that into a classifier:

> P(spam | words) is proportional to P(spam) × P(word₁ | spam) × P(word₂ | spam) × …

- **P(spam)** — how common spam is overall (the prior).
- **P(word | spam)** — how often each word appears in spam messages, learned from the training data.

The model computes this score for every class (spam and not spam) and picks the larger one.

## Why "naive"?

It assumes every word (feature) is **independent** of the others once you know the class. That is clearly false — "free" and "prize" often appear together. Surprisingly, Naive Bayes still works well, especially for **text**, because it only needs the **right ranking** of classes, not perfect probabilities.

## A tiny SMS spam filter

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

messages = [
    "Win a free phone now", "Claim your prize money today",
    "Meeting moved to 3pm", "Can you send the report",
    "Free airtime click this link", "Lunch at noon tomorrow?",
    "You won cash prize claim now", "Please review the budget file",
]
labels = [1, 1, 0, 0, 1, 0, 1, 0]          # 1 = spam

vec = CountVectorizer()                     # turns text into word counts
X_text = vec.fit_transform(messages)
print(vec.get_feature_names_out()[:10])

spam_filter = MultinomialNB().fit(X_text, labels)

new = ["Free prize waiting claim now", "Send me the meeting notes"]
print(spam_filter.predict(vec.transform(new)))
print(spam_filter.predict_proba(vec.transform(new)).round(3))

The first new message is classified as spam (probability about **0.99**), the second as normal (spam probability about **0.07**) — after seeing only eight examples!

`CountVectorizer` builds a **bag of words**: one column per word, counting how often it appears. Word order is ignored.

## Smoothing: the zero problem

If the word "airtime" never appeared in normal training messages, P(airtime | normal) would be 0 and would wipe out the whole product. Naive Bayes adds a small count to every word (**Laplace smoothing**, the `alpha=1.0` setting) so nothing is ever exactly zero.

## Naive Bayes for numbers

`GaussianNB` assumes each numeric feature follows a normal (bell) curve within each class.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
n = 2000
income = rng.lognormal(12.2, 0.45, n).round(-3)
loan = rng.uniform(20_000, 300_000, n).round(-3)
months = rng.integers(1, 61, n)
late = rng.poisson(0.6, n)
salary = rng.random(n) < 0.55
z = -3.2 + 2.5 * (loan / income) - 0.03 * months + 0.8 * late - 0.9 * salary
default = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
loans = pd.DataFrame({"income_rwf": income, "loan_rwf": loan, "months_customer": months,
                      "late_payments": late, "salaried": salary.astype(int),
                      "defaulted": default})
print(loans.shape, "default rate:", round(loans["defaulted"].mean(), 3))
loans.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = loans.drop(columns="defaulted")
y = loans["defaulted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB().fit(X_train, y_train)
print("Test accuracy:", round(nb.score(X_test, y_test), 3))

About **0.81** — lower than logistic regression (0.84), because income and loan size are not normally distributed and the model cannot combine them into the loan-to-income ratio that really drives risk.

## Which Naive Bayes?

| Version | Features | Typical use |
|---|---|---|
| MultinomialNB | Word counts | Spam filters, topic classification |
| BernoulliNB | Yes/no features (word present or not) | Short texts |
| GaussianNB | Numbers | Quick baseline on numeric data |

## Strengths and weaknesses

Naive Bayes trains in a blink, needs little data, and works well for text. Its probabilities are often too extreme (close to 0 or 1), and it cannot learn interactions between features.

## Key terms

- **Naive Bayes** — a classifier using Bayes' theorem with independent features.
- **Prior** — how common a class is before seeing the evidence.
- **Bag of words** — text represented by word counts, ignoring order.
- **Laplace smoothing** — adding a small count so no probability is zero.

**Takeaway:** Naive Bayes multiplies simple word (or feature) probabilities; it is a fast, strong baseline for text even though its independence assumption is false.

## Try it yourself

1. Add five more messages of your own (some in Kinyarwanda, for example "Watsindiye amafaranga") and retrain. Does it still work?
2. Try `BernoulliNB` on the same messages.
3. Print the words with the biggest difference in `spam_filter.feature_log_prob_` between the two classes. Which words are most "spammy"?
4. Explain to a friend why the model is called "naive".

---
Finished? Go back to [Machine Learning Fundamentals on WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*